# Fine-Tune LLaMA 3.2 3B for House Price Prediction

**Run this notebook in Google Colab with a T4 GPU**

This notebook fine-tunes `meta-llama/Llama-3.2-3B` using QLoRA (4-bit quantization + LoRA adapters) for house price prediction.

**Prerequisites:**
1. HuggingFace account with access to LLaMA 3.2 models
2. W&B account (project: `llama-3.2-housing-pricer`)

## Step 1: Install Dependencies

In [ ]:
!pip install -q --upgrade bitsandbytes==0.49 trl==0.25.1 transformers peft bitsandbytes accelerate trl datasets huggingface_hub wandb
!python --version

## Step 2: Login to HuggingFace and W&B

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
WANDB_API_KEY = userdata.get("WANDB_API_KEY")

login(token=HF_TOKEN)

In [ ]:
import wandb
wandb.login(key=WANDB_API_KEY)

## Step 3: Upload Training Data

Upload `housing_pricer_train_finetune.jsonl` and `housing_pricer_validation.jsonl` _(from week6)_ from your local machine, or run the cell below to create sample data.

In [ ]:
# Upload from local machine
from google.colab import files
uploaded = files.upload()  # Upload housing_pricer_train_finetune.jsonl and housing_pricer_validation.jsonl

## Step 4: Load Model with 4-bit Quantization

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig, setup_chat_format
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_name = "meta-llama/Llama-3.2-3B"

print(f"Loading {model_name} with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model.generation_config.pad_token_id = tokenizer.pad_token_id

model, tokenizer = setup_chat_format(model, tokenizer)

print(f"Model loaded. Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

# Test the loaded model

In [ ]:

SYSTEM_PROMPT = """
You are a real estate appraiser. Based on property details, estimate the market price.
You only respond with the price, no other text.
"""

test_description = "A 3 bedroom, 2.5 bathroom house with 2,100 square feet on a 0.3 acre lot located in Portland, Oregon, 97201."

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT

    },
    {
        "role": "user",
        "content": f"What is the estimated price of this property?\n\n{test_description}"

    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
  ).to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=10)

response = tokenizer.decode(outputs[0])
print(f"\nPredicted price: {response.split('assistant')[-1].strip()}")

## Step 5: Configure LoRA Adapters

In [ ]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
    bias="none"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")

## Step 6: Load Training Dataset

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset('json', data_files='housing_pricer_train_finetune.jsonl', split="train")
validation_dataset = load_dataset('json', data_files='housing_pricer_validation.jsonl', split="train")

print(f"Loaded {len(train_dataset)} training examples")
print("\nExample:")
print(train_dataset[0])

print(f"Loaded {len(validation_dataset)} validation examples")
print("\nExample:")
print(validation_dataset[0])

## Step 7: Train with W&B Logging

In [ ]:
from datetime import datetime

wandb.init(project="llama-3.2-housing-pricer", name="llama-3.2-3b-qlora")

training_args = SFTConfig(
    output_dir="./llama-house-pricer",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=100,
    save_total_limit=10,
    logging_steps=5,
    learning_rate=1e-4,
    weight_decay=0.001,
    fp16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.01,
    lr_scheduler_type="cosine",
    report_to="wandb",
    run_name=f"{datetime.now():%Y-%m-%d_%H.%M.%S}",
    max_length=128,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id="davemathews/housing_pricer",
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=100
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()

## Step 8: Save Model

In [ ]:
model.save_pretrained("./llama-house-pricer-final")
tokenizer.save_pretrained("./llama-house-pricer-final")
print("Model saved to ./llama-house-pricer-final")

## Step 9: Test the Fine-Tuned Model

In [ ]:

model.eval()

test_description = "A 1 bedroom, 1 bathroom house with 800 square feet"

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT

    },
    {
        "role": "user",
        "content": f"What is the estimated price of this property?\n\n{test_description}"

    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
  ).to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=7)

response = tokenizer.decode(outputs[0])
print(f"\nPredicted price: {response.split('assistant')[-1].strip()}")

In [ ]:
wandb.finish()
print("\nTraining complete! Check your results at: https://wandb.ai/")

## Step 10: Push to huggingface

In [ ]:
# Zip and download the model
trainer.model.push_to_hub("davemathews/housing_pricer", private=True)
